# Assignment 1

In this assignment, you'll be working with messy medical data and using regex to extract relevant infromation from the data. 

Each line of the `dates.txt` file corresponds to a medical note. Each note has a date that needs to be extracted, but each date is encoded in one of many formats.

The goal of this assignment is to correctly identify all of the different date variants encoded in this dataset and to properly normalize and sort the dates. 

Here is a list of some of the variants you might encounter in this dataset:
* 04/20/2009; 04/20/09; 4/20/09; 4/3/09
* Mar-20-2009; Mar 20, 2009; March 20, 2009;  Mar. 20, 2009; Mar 20 2009;
* 20 Mar 2009; 20 March 2009; 20 Mar. 2009; 20 March, 2009
* Mar 20th, 2009; Mar 21st, 2009; Mar 22nd, 2009
* Feb 2009; Sep 2009; Oct 2010
* 6/2008; 12/2009
* 2009; 2010

Once you have extracted these date patterns from the text, the next step is to sort them in ascending chronological order accoring to the following rules:
* Assume all dates in xx/xx/xx format are mm/dd/yy
* Assume all dates where year is encoded in only two digits are years from the 1900's (e.g. 1/5/89 is January 5th, 1989)
* If the day is missing (e.g. 9/2009), assume it is the first day of the month (e.g. September 1, 2009).
* If the month is missing (e.g. 2010), assume it is the first of January of that year (e.g. January 1, 2010).
* Watch out for potential typos as this is a raw, real-life derived dataset.

With these rules in mind, find the correct date in each note and return a pandas Series in chronological order of the original Series' indices. **This Series should be sorted by a tie-break sort in the format of ("extracted date", "original row number").**

For example if the original series was this:

    0    1999
    1    2010
    2    1978
    3    2015
    4    1985

Your function should return this:

    0    2
    1    4
    2    0
    3    1
    4    3

Your score will be calculated using [Kendall's tau](https://en.wikipedia.org/wiki/Kendall_rank_correlation_coefficient), a correlation measure for ordinal data.

*This function should return a Series of length 500 and dtype int.*

In [8]:
import pandas as pd

doc = []
with open('assets/dates.txt') as file:
    for line in file:
        doc.append(line)

df = pd.Series(doc)
df.head(10)

0         03/25/93 Total time of visit (in minutes):\n
1                       6/18/85 Primary Care Doctor:\n
2    sshe plans to move as of 7/8/71 In-Home Servic...
3                7 on 9/27/75 Audit C Score Current:\n
4    2/6/96 sleep studyPain Treatment Pain Level (N...
5                    .Per 7/06/79 Movement D/O note:\n
6    4, 5/18/78 Patient's thoughts about current su...
7    10/24/89 CPT Code: 90801 - Psychiatric Diagnos...
8                         3/7/86 SOS-10 Total Score:\n
9             (4/10/71)Score-1Audit C Score Current:\n
dtype: object

In [10]:
import re
def date_sorter():
    
    order = None
    # YOUR CODE HERE
    
    #extract these date patterns from the text
    #put regex in df and print row that is empty, keep updating code to match all regex
    
    #04/20/2009; 04/20/09; 4/20/09; 4/3/09
    df_1=df.str.findall(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}').to_frame(name="dates")
    #rearrange to mm/dd/yyyy format
    df_1['dates_str'] = df_1['dates'].str[0]
    df_1['dates_str'] = pd.to_datetime(df_1['dates_str']).dt.strftime('%m/%d/%Y')
    #print(df_1['dates_str'].to_list())
    
    
    #Mar-20-2009; Mar 20, 2009; March 20, 2009; Mar. 20, 2009; Mar 20 2009;
    df_2=df.str.findall(r'(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{2}.{,2}\d{4}').to_frame(name="dates")
    #rearrange to mm/dd/yyyy format
    df_2['dates_str'] = df_2['dates'].str[0]
    df_2['dates_str'] = pd.to_datetime(df_2['dates_str']).dt.strftime('%m/%d/%Y')
    #print(df_2['dates_str'].to_list())
    
    
    
    #20 Mar 2009; 20 March 2009; 20 Mar. 2009; 20 March, 2009
    df_3=df.str.findall(r'\d{2}\s(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* [12]\d{3}').to_frame(name="dates")
    #rearrange to mm/dd/yyyy format
    df_3['dates_str'] = df_3['dates'].str[0]
    df_3['dates_str'] = pd.to_datetime(df_3['dates_str']).dt.strftime('%m/%d/%Y')
    #print(df_3['dates_str'].to_list())
    
    
    #Mar 20th, 2009; Mar 21st, 2009; Mar 22nd, 2009, October. 11, 2013
    df_4=df.str.findall(r'(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*.? \d{1,2}.{,2}, \d{4}').to_frame(name="dates")
    #rearrange to mm/dd/yyyy format
    df_4['dates_str'] = df_4['dates'].str[0]
    df_4['dates_str'] = pd.to_datetime(df_4['dates_str']).dt.strftime('%m/%d/%Y')
    #print(df_4['dates_str'].to_list())
          
    #Feb 2009; Sep 2009; Oct 2010: should not be preceeded by number
    df_5=df.str.findall(r'(?<!\b\d{2} )(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]{,6}.? \d{4}').to_frame(name="dates")
    #rearrange to mm/dd/yyyy format
    df_5['dates_str'] = df_5['dates'].str[0]
    df_5['dates_str']=df_5['dates_str'].str.replace(r'Janaury', 'January', regex=True)
    df_5['dates_str']=df_5['dates_str'].str.replace(r'Decemeber', 'December', regex=True)
    df_5['dates_str'] = pd.to_datetime(df_5['dates_str']).dt.strftime('%m/%d/%Y')
    #print(df_5['dates_str'].to_list())
    
    
    #6/2008; 12/2009: only should not be preceeded by any other date
    df_6=df.str.findall(r'(?<!/)\b\d{1,2}/\d{4}').to_frame(name="dates")
    #rearrange to mm/dd/yyyy format
    df_6['dates_str'] = df_6['dates'].str[0]
    df_6['dates_str'] = pd.to_datetime(df_6['dates_str']).dt.strftime('%m/%d/%Y')
    #print(df_6['dates_str'].to_list())
    
    df_6_1=df.str.findall(r'(?<=[a-zA-Z])\d{1,2}/\d{4}').to_frame(name="dates")
    #rearrange to mm/dd/yyyy format
    df_6_1['dates_str'] = df_6_1['dates'].str[0]
    df_6_1['dates_str'] = pd.to_datetime(df_6_1['dates_str']).dt.strftime('%m/%d/%Y')
    #print(df_6_1['dates_str'].to_list())
       
    #combine all df_nums and check if total length matches
    #find the correct date in each note and return a pandas Series in chronological order of the original Series' indices. 
    
    df_1.dropna(inplace=True)
    #print(df_1['dates_str'])
    
    df_2.dropna(inplace=True)
    #print(df_2['dates_str'])
    
    df_3.dropna(inplace=True)
    #print(df_3['dates_str'])
    
    df_4.dropna(inplace=True)
    #print(df_4['dates_str'])
    
    df_5.dropna(inplace=True)
    #print(df_5['dates_str'])
    
    df_6.dropna(inplace=True)
    #print(df_6['dates_str'])
    
    df_6_1.dropna(inplace=True)
    #print(df_6['dates_str'])
    
    #df_7_1.dropna(inplace=True)
    #print(df_7_1['dates_str'])
    
    #This Series should be sorted by a tie-break sort in the format of ("extracted date", "original row number")   
    #the next step is to sort them in ascending chronological order
    df_dates = pd.concat([df_1, df_2], ignore_index=False)
    df_dates = pd.concat([df_dates, df_3], ignore_index=False)
    df_dates = pd.concat([df_dates, df_4], ignore_index=False)
    df_dates = pd.concat([df_dates, df_5], ignore_index=False)
    df_dates = pd.concat([df_dates, df_6], ignore_index=False)
    df_dates = pd.concat([df_dates, df_6_1], ignore_index=False)
    #print(df_dates)
          
    #print(df_dates.index.tolist())
    index_list=[]
    data_list=[]
    for ind in range(500):
        if ind not in df_dates.index.tolist():
            #get 4 digit numbers from given list of indices
            cell_value = str(df.loc[ind])
            match = re.search(r'\d{4}', cell_value)
            index_list.append(ind)
            data_list.append(match.group())
            #print(ind,match.group())
    
    df_7_1 = pd.DataFrame(data=data_list, index=index_list, columns=["dates_str"])
    df_7_1['dates_str'] = pd.to_datetime(df_7_1['dates_str']).dt.strftime('%m/%d/%Y')       
    #print(df_7_1['dates_str'].to_list())
    
    
    df_dates = pd.concat([df_dates, df_7_1], ignore_index=False)
    #print(df_dates)
    #df_dates = df_dates.drop(194)
    #find duplicated index
    #ind_list = df_dates.index.to_list()
    #for ind in range(500):
        #ind_count=ind_list.count(ind)
        #if ind_count >1:
            #print(ind)
        
    #print(ind_list)
    #drop duplicates
    df_cleaned = df_dates[~df_dates.index.duplicated(keep='first')]  
    #print(df_cleaned)
    
    #for ind in range(500):
        #if ind not in df_dates.index.tolist():
            #print(ind)
    df_cleaned = df_cleaned.drop(columns='dates')
    #print(df_cleaned)
    
    df_cleaned['dates_str'] = pd.to_datetime(df_cleaned['dates_str'])
    #print(filtered_df)
    
    # 1. Create a boolean mask for rows where the year is greater than 2025
    mask = df_cleaned['dates_str'].dt.year > 2025

    # 2. Subtract 100 years from only those filtered rows
    df_cleaned.loc[mask, 'dates_str'] = df_cleaned.loc[mask, 'dates_str'] - pd.DateOffset(years=100)
    
    #print(df_cleaned)
    df_sorted = df_cleaned.sort_values(by='dates_str', kind="stable")
    #print(df_sorted.loc[427])
    #print(df_sorted.loc[231])
    
    order = pd.Series(df_sorted.index.to_list(), index=range(500))
    #return 0
    #raise NotImplementedError()
    return order # Your answer here

print(date_sorter())


0        9
1       84
2        2
3       53
4       28
      ... 
495    427
496    141
497    186
498    161
499    413
Length: 500, dtype: int64
